In [ ]:
# --- Paths (repo-relative; this notebook runs from notebooks/) ---
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
DATA      = PROJECT_ROOT / 'data'
RAW       = DATA / 'raw'          # external source data
ANNOTATED = DATA / 'annotated'    # pipeline-derived tables
FIGURES   = PROJECT_ROOT / 'figures'
# NOTE: some cells below still read AKEY-cluster paths (/scratch/gpfs/... ,
# /projects/AKEY/...) for inputs not mirrored in this repo (goatools GO files,
# UniProt idmapping .parquet, HumanTFs .csv). Those cells only run on the cluster.


# Q4 IDR-variation GO enrichment and canonical disorder analysis
This notebook tests whether genes in the highest quartile of isoform-level IDR
variation (Q4) are enriched for Gene Ontology (GO) Biological Process terms,
analyzed separately for transcription factors (TFs) and Non-TFs.

## Figures
1. **GO levels 2 and 3** — TF and Non-TF subplots in one figure  
2. **GO levels 3 and 4** — TF and Non-TF subplots in one figure  
3. **Full GO Biological Process** — TF and Non-TF subplots in one figure  
4. **Canonical disorder content** — TF and Non-TF side-by-side histograms  

## 1. Setup and file paths

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import textwrap

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import mannwhitneyu

from goatools.anno.gaf_reader import GafReader
from goatools.go_enrichment import GOEnrichmentStudy
from goatools.obo_parser import GODag

# Input files
FIG1A_PATH = Path("../data/annotated/fig1a_quartiles.csv")
GAF_PATH = Path("/projects/AKEY/akey_vol1/software/goatools/goa_human.gaf")
OBO_PATH = Path("/projects/AKEY/akey_vol1/software/goatools/go-basic.obo")

# Output directory
OUTPUT_DIR = Path("../figures/go_enrichment")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Shared Figure Settings
# ------------------------------------------------------------------
TOP_K_GO_TERMS = 20 # number of go terms per plot
COUNT_LEGEND_SIZES = [5, 10, 20, 50] # count legend sizes
HISTOGRAM_BINS = np.arange(0, 105, 5) # increments of 5

mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 9.5,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("Output directory:", OUTPUT_DIR) # directory saving in

## 2. Load the quartile and GO data

In [ ]:
fig1a_quartiles_df = pd.read_csv(FIG1A_PATH, low_memory=False)

required_columns = {"base_accession","tf_group","pct_change_idr","pct_change_idr_quartile",}

fig1a_quartiles_df = (
    fig1a_quartiles_df
    .dropna(subset=["base_accession", "tf_group"])
    .drop_duplicates(["base_accession", "tf_group"])
    .copy()
)

associations = GafReader(str(GAF_PATH)).read_gaf()
obodag = GODag(str(OBO_PATH), optional_attrs={"relationship"})

print("Quartile dataframe shape:", fig1a_quartiles_df.shape)
display(fig1a_quartiles_df["tf_group"].value_counts())
display(fig1a_quartiles_df["pct_change_idr_quartile"].value_counts())
print("Genes with GO annotations:", len(associations))
print("GO terms in ontology:", len(obodag))

## 3. Define the study and background cohorts

Quartiles were assigned separately within TFs and Non-TFs. For each GO analysis,
we test Q4 genes against their own group-specific background:

- **TF study:** Q4 TF genes versus all TF genes  
- **Non-TF study:** Q4 Non-TF genes versus all Non-TF genes

In [ ]:
gene_quartile_df = (
    fig1a_quartiles_df
    .dropna(subset=["base_accession","tf_group","pct_change_idr","pct_change_idr_quartile",])
    .drop_duplicates(["base_accession", "tf_group"]).copy())

q4_range_summary = (
    gene_quartile_df
    .query("pct_change_idr_quartile == 'Q4'")
    .groupby("tf_group", observed=True)["pct_change_idr"]
    .agg(n_genes="size",q4_min="min",q4_median="median",q4_mean="mean",q4_max="max",).round(3))

display(q4_range_summary)

for group, row in q4_range_summary.iterrows():
    print(
        f"{group} Q4: {row['q4_min']:.3f}% to {row['q4_max']:.3f}% "
        f"ΔIDR (median={row['q4_median']:.3f}%, n={int(row['n_genes'])})")

# Check whether raw-value ties overlap the Q3/Q4 boundary.
boundary_check = (
    q4_range_df.query("pct_change_idr_quartile in ['Q1','Q2','Q3', 'Q4']")
    .groupby(["tf_group", "pct_change_idr_quartile"], observed=True)["pct_change_idr"]
    .agg(["min", "max"]))

display(boundary_check)

tf_background_df = gene_quartile_df.query("tf_group == 'TF'").copy()
tf_q4_study_df = gene_quartile_df.query("tf_group == 'TF' and pct_change_idr_quartile == 'Q4'").copy()

nontf_background_df = gene_quartile_df.query("tf_group == 'Non-TF'").copy()
nontf_q4_study_df = gene_quartile_df.query("tf_group == 'Non-TF' and pct_change_idr_quartile == 'Q4'").copy()

tf_population_ids = set(tf_background_df["base_accession"])
tf_study_ids = set(tf_q4_study_df["base_accession"])

nontf_population_ids = set(nontf_background_df["base_accession"])
nontf_study_ids = set(nontf_q4_study_df["base_accession"])

assert tf_study_ids <= tf_population_ids
assert nontf_study_ids <= nontf_population_ids

print(f"All TF background genes: {len(tf_population_ids):,}")
print(f"Q4 TF study genes: {len(tf_study_ids):,}")
print(f"All Non-TF background genes: {len(nontf_population_ids):,}")
print(f"Q4 Non-TF study genes: {len(nontf_study_ids):,}")

## 4. GO analysis and Plotting functions

In [ ]:
def get_bp_terms_at_levels(godag, levels):
    """Return Biological Process GO IDs whose ontology level is in `levels`."""
    levels = set(levels)
    return {
        go_id
        for go_id, term in godag.items()
        if getattr(term, "level", -1) in levels
        and term.namespace == "biological_process"
    }


def project_associations_to_terms(associations, godag, selected_terms):
    """
    Project detailed gene-to-GO annotations onto a selected set of ancestor terms.
    """
    projected = defaultdict(set)

    for gene_id, go_terms in associations.items():
        for go_id in go_terms:
            if go_id not in godag:
                continue

            ancestors = godag[go_id].get_all_parents()
            ancestors.add(go_id)

            matches = ancestors.intersection(selected_terms)
            if matches:
                projected[gene_id].update(matches)

    return dict(projected)


def run_go_enrichment(
    study_ids_raw,
    population_ids_raw,
    associations,
    godag,
    *,
    allowed_terms=None,
    namespace="BP",
    propagate_counts=False,
    label="GO enrichment",
):
    """
    Run GO enrichment and return all tested terms in a tidy dataframe.
    """
    annotated_ids = set(associations)

    population_ids = set(population_ids_raw) & annotated_ids
    study_ids = set(study_ids_raw) & population_ids

    if not study_ids:
        raise ValueError(f"{label}: no study genes have usable GO annotations.")
    if not population_ids:
        raise ValueError(f"{label}: no population genes have usable GO annotations.")

    print(f"\n{label}")
    print(f"Study mapped: {len(study_ids):,}/{len(study_ids_raw):,}")
    print(
        f"Population mapped: "
        f"{len(population_ids):,}/{len(population_ids_raw):,}"
    )

    goea = GOEnrichmentStudy(
        population_ids,
        associations,
        godag,
        propagate_counts=propagate_counts,
        methods=["fdr_bh"],
        alpha=0.05,
    )

    results = goea.run_study(study_ids)
    rows = []

    for result in results:
        if result.NS != namespace:
            continue
        if allowed_terms is not None and result.GO not in allowed_terms:
            continue

        term = godag.get(result.GO)
        study_in, study_n = result.ratio_in_study
        pop_in, pop_n = result.ratio_in_pop

        study_ratio = study_in / study_n
        pop_ratio = pop_in / pop_n
        fold_enrichment = study_ratio / pop_ratio if pop_ratio > 0 else np.nan

        rows.append({
            "GO": result.GO,
            "name": result.name,
            "NS": result.NS,
            "study_in": study_in,
            "study_n": study_n,
            "pop_in": pop_in,
            "pop_n": pop_n,
            "study_ratio": study_ratio,
            "pop_ratio": pop_ratio,
            "fold_enrichment": fold_enrichment,
            "direction": "enriched" if fold_enrichment > 1 else "depleted",
            "p_uncorrected": result.p_uncorrected,
            "p_fdr_bh": result.p_fdr_bh,
            "level": term.level if term else np.nan,
            "depth": term.depth if term else np.nan,
        })

    output = pd.DataFrame(rows)

    if output.empty:
        print("No GO terms were returned.")
        return output

    smallest_positive = np.nextafter(0, 1)

    output["neglog10_raw_p"] = -np.log10(
        output["p_uncorrected"].clip(lower=smallest_positive)
    )
    output["neglog10_fdr"] = -np.log10(
        output["p_fdr_bh"].clip(lower=smallest_positive)
    )
    output["nominal"] = output["p_uncorrected"] < 0.05
    output["significant_fdr"] = output["p_fdr_bh"] < 0.05

    output = (
        output
        .sort_values(
            ["p_fdr_bh", "p_uncorrected", "fold_enrichment"],
            ascending=[True, True, False],
        )
        .reset_index(drop=True)
    )

    print(f"Terms tested: {len(output):,}")
    print(f"Nominal p < 0.05: {int(output['nominal'].sum()):,}")
    print(f"FDR < 0.05: {int(output['significant_fdr'].sum()):,}")

    display(output.head(20))
    return output


def prepare_top_terms(results_df, top_k=TOP_K_GO_TERMS):
    """
    Filter to enriched terms and select the top `top_k` by raw p-value.
    """
    plot_df = results_df.query("direction == 'enriched'").copy()

    if plot_df.empty:
        return pd.DataFrame()

    plot_df["GeneRatio"] = plot_df["study_in"] / plot_df["study_n"]
    plot_df["Count"] = plot_df["study_in"]

    top = (plot_df.sort_values(["p_uncorrected", "fold_enrichment"], ascending=[True, False]).head(top_k).copy())

    top["term"] = (top["name"]
        .str.replace(r"\s+", " ", regex=True)
        .apply(lambda name: "\n".join(textwrap.wrap(name, width=40)))
        + "  [L"+ top["level"].astype("Int64").astype(str)+ "]")

    top = (top.sort_values("neglog10_raw_p", ascending=True).reset_index(drop=True))

    return top

In [ ]:
def draw_go_row(ax_left, ax_right, top_df, row_label):
    """
    Draw one row of the combined TF/Non-TF GO figure.
    """
    y = np.arange(len(top_df))

    scatter = ax_left.scatter(
        top_df["GeneRatio"],y,s=6 * top_df["Count"],c=top_df["neglog10_raw_p"],
        cmap="plasma",edgecolors="white",linewidths=0.7,alpha=0.95,zorder=3,)
    ax_left.set_yticks(y)
    ax_left.set_yticklabels(top_df["term"])
    ax_left.invert_yaxis()
    ax_left.set_xlabel("Gene ratio")
    ax_left.set_title(f"{row_label} · Enriched Biological Process terms",fontweight="bold",loc="left",)
    ax_left.set_xlim(0,max(0.10, top_df["GeneRatio"].max() * 1.18),)
    ax_left.grid(axis="x", linestyle="--", linewidth=0.7, alpha=0.25, zorder=0)
    ax_left.spines["left"].set_visible(False)

    mapped_study_n = int(top_df["study_n"].iloc[0])
    mapped_background_n = int(top_df["pop_n"].iloc[0])

    ax_left.text(0.0,1.05,
        f"mapped study n = {mapped_study_n:,}, mapped background n = {mapped_background_n:,}",
        transform=ax_left.transAxes,ha="left",va="bottom",fontsize=9.5,color="0.35",)

    colorbar = plt.colorbar(scatter, ax=ax_left, fraction=0.042, pad=0.025)
    colorbar.set_label(r"$-\log_{10}$(raw p-value)")

    count_handles = [
        Line2D([0],[0],marker="o",linestyle="none",label=str(count),markerfacecolor="gray",
               markeredgecolor="white",markeredgewidth=0.7,markersize=np.sqrt(6 * count),alpha=0.9,)
        for count in COUNT_LEGEND_SIZES
    ]

    count_legend = ax_left.legend(
        handles=count_handles,
        title= "Count",
        frameon=True,
        fancybox=True,
        framealpha=0.95,
        edgecolor="0.85",
        loc="upper right",
        labelspacing=1.15,
        borderpad=0.8,
    )
    count_legend.get_title().set_fontweight("bold")

    cycle_colors = mpl.rcParams["axes.prop_cycle"].by_key()["color"]
    raw_color = cycle_colors[0]
    fdr_color = cycle_colors[1]

    for row_index in range(len(top_df)):
        ax_right.plot(
            [
                top_df["neglog10_fdr"].iloc[row_index],
                top_df["neglog10_raw_p"].iloc[row_index],
            ],
            [y[row_index], y[row_index]],
            color="0.72",
            linewidth=1.5,
            alpha=0.8,
            solid_capstyle="round",
            zorder=1,
        )

    ax_right.scatter(
        top_df["neglog10_raw_p"],
        y,
        s=82,
        label="Raw p-value",
        color=raw_color,
        edgecolors="white",
        linewidths=0.7,
        alpha=0.95,
        zorder=3,
    )
    ax_right.scatter(
        top_df["neglog10_fdr"],
        y,
        s=82,
        label="FDR-adjusted p-value",
        color=fdr_color,
        edgecolors="white",
        linewidths=0.7,
        alpha=0.95,
        zorder=3,
    )

    threshold = -np.log10(0.05)
    x_max = max(
        threshold * 1.15,
        top_df["neglog10_raw_p"].max() * 1.15,
        top_df["neglog10_fdr"].max() * 1.15,
        1.5,
    )

    ax_right.axvspan(threshold, x_max, alpha=0.04, zorder=0)
    ax_right.axvline(
        threshold,
        linestyle="--",
        linewidth=1.5,
        color="black",
        alpha=0.8,
        label="0.05 threshold",
        zorder=2,
    )

    ax_right.set_xlim(-0.1, x_max)
    ax_right.set_xlabel(r"$-\log_{10}$(p-value)")
    ax_right.set_title(
        f"{row_label} · Raw P-value versus FDR correction",
        fontweight="bold",
        loc="left",
    )
    ax_right.grid(axis="x", linestyle="--", linewidth=0.7, alpha=0.25, zorder=0)
    ax_right.spines["left"].set_visible(False)
    ax_right.tick_params(axis="y", left=False, labelleft=False)
    ax_right.legend(
        frameon=True,
        fancybox=True,
        framealpha=0.95,
        edgecolor="0.85",
        loc="lower right",
    )


def plot_go_tf_nontf_figure(
    tf_results_df,
    nontf_results_df,
    *,
    title,
    subtitle,
    output_path,
    top_k=TOP_K_GO_TERMS,
):
    """
    Create a 2x2 combined GO figure:
        row 1 = TF analysis
        row 2 = Non-TF analysis
        col 1 = GO dot plot
        col 2 = raw versus FDR-adjusted p-values
    """
    top_tf = prepare_top_terms(tf_results_df, top_k=top_k)
    top_nontf = prepare_top_terms(nontf_results_df, top_k=top_k)

    if top_tf.empty and top_nontf.empty:
        print(f"{title}: neither TF nor Non-TF has enriched terms to plot.")
        return None, top_tf, top_nontf
    if top_tf.empty or top_nontf.empty:
        raise ValueError(
            "Expected both TF and Non-TF enriched terms to create the combined figure."
        )

    nrows_total = len(top_tf) + len(top_nontf)
    fig_height = max(11, 0.58 * nrows_total)

    fig, axes = plt.subplots(
        nrows=2,
        ncols=2,
        figsize=(19, fig_height),
        gridspec_kw={
            "width_ratios": [1.2, 1],
            "hspace": 0.2,
            "wspace": 0.10,
        },
    )

    draw_go_row(axes[0, 0], axes[0, 1], top_tf, "A. TFs")
    draw_go_row(axes[1, 0], axes[1, 1], top_nontf, "B. Non-TFs")

    fig.suptitle(
        title,
        fontsize=17,
        fontweight="bold",
        y=0.94,
    )
    fig.text(
        0.5,
        0.92,
        subtitle,
        ha="center",
        va="top",
        fontsize=11,
        color="0.35",
    )

    plt.tight_layout(rect=[0, 0.04, 1, 0.95])

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, format="pdf", bbox_inches="tight")

    print("Saved PDF:", output_path)
    plt.show()

    return fig, top_tf, top_nontf

## 5. Figure 1 — GO Biological Process levels 2 and 3

Detailed GO annotations are projected onto Biological Process terms whose ontology
level is exactly 2 or 3. Both the TF and Non-TF Q4 cohorts are analyzed using
their own group-specific backgrounds and shown together in a single multi-panel figure.

In [ ]:
levels_23 = {2, 3}
level23_terms = get_bp_terms_at_levels(obodag, levels_23)
associations_level23 = project_associations_to_terms(
    associations,
    obodag,
    level23_terms,
)

print("Selected BP terms:", len(level23_terms))
print("Level counts:", Counter(obodag[term].level for term in level23_terms))
print("Genes mapped to levels 2/3:", len(associations_level23))

out_tf_q4_level23 = run_go_enrichment(
    tf_study_ids,
    tf_population_ids,
    associations_level23,
    obodag,
    allowed_terms=level23_terms,
    namespace="BP",
    propagate_counts=False,
    label="Q4 TFs vs all TFs: GO BP levels 2 and 3",
)

out_nontf_q4_level23 = run_go_enrichment(
    nontf_study_ids,
    nontf_population_ids,
    associations_level23,
    obodag,
    allowed_terms=level23_terms,
    namespace="BP",
    propagate_counts=False,
    label="Q4 Non-TFs vs all Non-TFs: GO BP levels 2 and 3",
)

#out_tf_q4_level23.to_csv(
#    OUTPUT_DIR / "Q4_TF_GO_BP_levels_2_3.csv",
#    index=False,
#)
#out_nontf_q4_level23.to_csv(
#    OUTPUT_DIR / "Q4_NonTF_GO_BP_levels_2_3.csv",
#    index=False,
#)

fig_go_23, top_go_23_tf, top_go_23_nontf = plot_go_tf_nontf_figure(
    out_tf_q4_level23,
    out_nontf_q4_level23,
    title="GO enrichment of high-IDR-variation genes",
    subtitle="Biological Process levels 2 and 3",
    output_path=OUTPUT_DIR / "Q4_TF_and_NonTF_GO_BP_levels_2_3.pdf",
)

## 6. Figure 2 — GO Biological Process levels 3 and 4

This analysis uses the same cohort definitions, test, and figure layout as
Figure 1, but projects annotations onto terms whose ontology level is exactly
3 or 4.

In [ ]:
levels_34 = {3, 4}
level34_terms = get_bp_terms_at_levels(obodag, levels_34)
associations_level34 = project_associations_to_terms(
    associations,
    obodag,
    level34_terms,
)

print("Selected BP terms:", len(level34_terms))
print("Level counts:", Counter(obodag[term].level for term in level34_terms))
print("Genes mapped to levels 3/4:", len(associations_level34))

out_tf_q4_level34 = run_go_enrichment(
    tf_study_ids,
    tf_population_ids,
    associations_level34,
    obodag,
    allowed_terms=level34_terms,
    namespace="BP",
    propagate_counts=False,
    label="Q4 TFs vs all TFs: GO BP levels 3 and 4",
)

out_nontf_q4_level34 = run_go_enrichment(
    nontf_study_ids,
    nontf_population_ids,
    associations_level34,
    obodag,
    allowed_terms=level34_terms,
    namespace="BP",
    propagate_counts=False,
    label="Q4 Non-TFs vs all Non-TFs: GO BP levels 3 and 4",
)

#out_tf_q4_level34.to_csv(
#    OUTPUT_DIR / "Q4_TF_GO_BP_levels_3_4.csv",
#    index=False,
#)
#out_nontf_q4_level34.to_csv(
#    OUTPUT_DIR / "Q4_NonTF_GO_BP_levels_3_4.csv",
#    index=False,
#)

fig_go_34, top_go_34_tf, top_go_34_nontf = plot_go_tf_nontf_figure(
    out_tf_q4_level34,
    out_nontf_q4_level34,
    title="GO enrichment of high-IDR-variation genes",
    subtitle="Biological Process levels 3 and 4",
    output_path=OUTPUT_DIR / "Q4_TF_and_NonTF_GO_BP_levels_3_4.pdf",
)

## 7. Figure 3 — Full GO Biological Process enrichment

The full analysis uses the detailed GAF annotations directly. GOATOOLS propagates
counts to ancestor terms, allowing specific and broad Biological Process terms
to be tested together. Both TF and Non-TF analyses are included in the same
standardized multi-panel figure.

In [ ]:
out_tf_q4_full_go = run_go_enrichment(
    tf_study_ids,
    tf_population_ids,
    associations,
    obodag,
    allowed_terms=None,
    namespace="BP",
    propagate_counts=True,
    label="Q4 TFs vs all TFs: full GO Biological Process",
)

out_nontf_q4_full_go = run_go_enrichment(
    nontf_study_ids,
    nontf_population_ids,
    associations,
    obodag,
    allowed_terms=None,
    namespace="BP",
    propagate_counts=True,
    label="Q4 Non-TFs vs all Non-TFs: full GO Biological Process",
)

#out_tf_q4_full_go.to_csv(
#    OUTPUT_DIR / "Q4_TF_full_GO_BP.csv",
#    index=False,
#)
#out_nontf_q4_full_go.to_csv(
#    OUTPUT_DIR / "Q4_NonTF_full_GO_BP.csv",
#    index=False,
#)

fig_go_full, top_go_full_tf, top_go_full_nontf = plot_go_tf_nontf_figure(
    out_tf_q4_full_go,
    out_nontf_q4_full_go,
    title="Full GO enrichment of high-IDR-variation genes",
    subtitle="Complete Biological Process ontology",
    output_path=OUTPUT_DIR / "Q4_TF_and_NonTF_full_GO_BP.pdf",
)

## 8. Figure 4 — Canonical disorder content in Q4 versus other genes

The left panel compares Q4 TFs with TFs in Q1–Q3. The right panel performs the
same comparison within Non-TFs. Each histogram is normalized to the percentage
of genes within its own group, so unequal group sizes do not dominate the visual
comparison.

In [ ]:
def identify_canonical_idr_column(dataframe):
    """
    Identify the column containing canonical percent-IDR values.
    """
    if "canonical_pct_idr" in dataframe.columns:
        return "canonical_pct_idr"

    if "pct_idr" not in dataframe.columns:
        raise KeyError(
            "Expected either 'canonical_pct_idr' or 'pct_idr' in the quartile table."
        )

    if "is_canonical" in dataframe.columns:
        canonical_mask = (
            dataframe["is_canonical"]
            .astype(str)
            .str.lower()
            .isin(["true", "1"])
        )
        if not canonical_mask.all():
            raise ValueError(
                "Some rows are not canonical, so 'pct_idr' cannot safely be used "
                "as canonical disorder content."
            )

    return "pct_idr"


def summarize_q4_comparison(group_df, canonical_column):
    """Return Q4/other values and a compact Mann–Whitney summary."""
    clean = (
        group_df
        .dropna(subset=[canonical_column, "pct_change_idr_quartile"])
        .copy()
    )

    q4_values = clean.loc[
        clean["pct_change_idr_quartile"].eq("Q4"),
        canonical_column,
    ].astype(float)

    other_values = clean.loc[
        ~clean["pct_change_idr_quartile"].eq("Q4"),
        canonical_column,
    ].astype(float)

    if q4_values.empty or other_values.empty:
        raise ValueError("Both Q4 and Q1–Q3 values are required for comparison.")

    u_statistic, p_value = mannwhitneyu(
        q4_values,
        other_values,
        alternative="two-sided",
    )

    rank_biserial = (
        2 * u_statistic / (len(q4_values) * len(other_values))
    ) - 1

    summary = {
        "q4_n": len(q4_values),
        "other_n": len(other_values),
        "q4_mean": q4_values.mean(),
        "other_mean": other_values.mean(),
        "q4_median": q4_values.median(),
        "other_median": other_values.median(),
        "median_difference": q4_values.median() - other_values.median(),
        "mannwhitney_u": u_statistic,
        "p_value": p_value,
        "rank_biserial": rank_biserial,
    }

    return q4_values, other_values, summary


canonical_column = identify_canonical_idr_column(gene_quartile_df)
print("Canonical disorder column:", canonical_column)

group_specs = [
    ("TF", "A. Transcription factors"),
    ("Non-TF", "B. Non-transcription factors"),
]

cycle_colors = mpl.rcParams["axes.prop_cycle"].by_key()["color"]
q4_color = cycle_colors[0]
other_color = cycle_colors[1]

fig, axes = plt.subplots(
    ncols=2,
    figsize=(15, 5.8),
    sharex=True,
    sharey=True,
)

histogram_summaries = []

for axis, (group_name, panel_title) in zip(axes, group_specs):
    group_df = gene_quartile_df.query("tf_group == @group_name").copy()

    q4_values, other_values, summary = summarize_q4_comparison(
        group_df,
        canonical_column,
    )

    histogram_summaries.append({
        "tf_group": group_name,
        **summary,
    })

    q4_weights = np.full(len(q4_values), 100 / len(q4_values))
    other_weights = np.full(len(other_values), 100 / len(other_values))

    axis.hist(
        q4_values,
        bins=HISTOGRAM_BINS,
        weights=q4_weights,
        histtype="step",
        linewidth=2.4,
        color=q4_color,
        label=f"Q4 (n={len(q4_values):,})",
    )
    axis.hist(
        other_values,
        bins=HISTOGRAM_BINS,
        weights=other_weights,
        histtype="step",
        linewidth=2.4,
        color=other_color,
        label=f"Q1–Q3 (n={len(other_values):,})",
    )

    axis.axvline(
        q4_values.median(),
        color=q4_color,
        linestyle="-",
        linewidth=1.6,
        alpha=0.9,
    )
    axis.axvline(
        other_values.median(),
        color=other_color,
        linestyle="--",
        linewidth=1.6,
        alpha=0.9,
    )

    axis.set_title(
        f"{panel_title}\nMann–Whitney p = {summary['p_value']:.2g}",
        fontweight="bold",
    )
    axis.set_xlabel("Canonical disorder content (%IDR)")
    axis.set_xlim(0, 100)
    axis.grid(axis="y", linestyle="--", linewidth=0.7, alpha=0.25)
    axis.legend(
        frameon=True,
        fancybox=True,
        framealpha=0.95,
        edgecolor="0.85",
    )

    annotation = (
        f"Median Q4: {summary['q4_median']:.1f}%\n"
        f"Median Q1–Q3: {summary['other_median']:.1f}%\n"
        f"Δ median: {summary['median_difference']:+.1f} pp"
    )
    axis.text(
        0.98,
        0.96,
        annotation,
        transform=axis.transAxes,
        ha="right",
        va="top",
        fontsize=9.5,
        bbox={
            "boxstyle": "round,pad=0.4",
            "facecolor": "white",
            "edgecolor": "0.85",
            "alpha": 0.95,
        },
    )

axes[0].set_ylabel("Genes within group (%)")

fig.suptitle(
    "Canonical disorder content by IDR-variation quartile",
    fontsize=16,
    fontweight="bold",
    y=1.02,
)

plt.tight_layout(rect=[0, 0.04, 1, 0.97])

canonical_pdf = OUTPUT_DIR / "Q4_canonical_disorder_TF_and_NonTF.pdf"
fig.savefig(canonical_pdf, format="pdf", bbox_inches="tight")

print("Saved PDF:", canonical_pdf)
plt.show()

canonical_summary_df = pd.DataFrame(histogram_summaries)
display(canonical_summary_df)

#canonical_summary_df.to_csv(
#    OUTPUT_DIR / "Q4_canonical_disorder_summary.csv",
#    index=False,
#)

## 9. Output manifest

In [ ]:
expected_outputs = [
    OUTPUT_DIR / "Q4_TF_and_NonTF_GO_BP_levels_2_3.pdf",
    OUTPUT_DIR / "Q4_TF_and_NonTF_GO_BP_levels_3_4.pdf",
    OUTPUT_DIR / "Q4_TF_and_NonTF_full_GO_BP.pdf",
    OUTPUT_DIR / "Q4_canonical_disorder_TF_and_NonTF.pdf",
    OUTPUT_DIR / "Q4_TF_GO_BP_levels_2_3.csv",
    OUTPUT_DIR / "Q4_NonTF_GO_BP_levels_2_3.csv",
    OUTPUT_DIR / "Q4_TF_GO_BP_levels_3_4.csv",
    OUTPUT_DIR / "Q4_NonTF_GO_BP_levels_3_4.csv",
    OUTPUT_DIR / "Q4_TF_full_GO_BP.csv",
    OUTPUT_DIR / "Q4_NonTF_full_GO_BP.csv",
    OUTPUT_DIR / "Q4_TF_full_GO_development_related_terms.csv",
    OUTPUT_DIR / "Q4_NonTF_full_GO_development_related_terms.csv",
    OUTPUT_DIR / "Q4_canonical_disorder_summary.csv",
]

print("Expected outputs:")
for output_file in expected_outputs:
    status = "created" if output_file.exists() else "not yet created"
    print(f"  [{status}] {output_file}")